In [18]:
import pandas as pd
from pathlib import Path
import json


In [ ]:
repo_root = Path.cwd().parent.parent.parent

df_urls = pd.read_json(
    repo_root/"dataset"/"urls.jsonl.gz",
    lines=True,
    compression="gzip", 
    nrows = 4000000)

df_meta = pd.read_json(
    repo_root/"dataset"/"zip"/"yt_metadata_en.jsonl.gz",
    lines=True,
    compression="gzip", 
    nrows = 4000000)

df_merged = df_urls.merge(df_meta, on="display_id", how="inner")

print("Merged columns:", df_merged.columns.tolist())

Merged columns: ['display_id', 'urls', 'categories', 'channel_id', 'crawl_date', 'description', 'dislike_count', 'duration', 'like_count', 'tags', 'title', 'upload_date', 'view_count']


In [20]:
df_merged['upload_date'] = pd.to_datetime(df_merged['upload_date'], errors='coerce')
df_merged['year'] = df_merged['upload_date'].dt.year

#remove all blank categories
df_merged = df_merged[
    (df_merged['categories'].notna()) & 
    (df_merged['categories'] != '')
].copy()

In [21]:
#CHECKING THE OUTPUTS 
#testing the output
'''
print("Sample merged data:")
print(df_merged[["display_id", "title", "categories", "upload_date", "urls"]].head())
'''

#Test: find videos w/ patreon links
'''
patreon_videos = df_merged[
    df_merged["urls"].apply(lambda urls: any("patreon" in url.lower() for url in urls))
]
print(f"\n\nFound {len(patreon_videos)} videos with Patreon links")
print(patreon_videos[["title", "categories", "urls"]].head())
'''

# Date check 
'''
print(df_merged['upload_date'].head(5))
print(df_merged['upload_date'].astype(str).head(20).tolist())
'''

"\nprint(df_merged['upload_date'].head(5))\nprint(df_merged['upload_date'].astype(str).head(20).tolist())\n"

In [22]:
print(f"The merge gave us {len(df_merged):,} records, {len(df_urls) - len(df_merged):,} records non matched ({(1 - len(df_merged)/len(df_urls))*100:.1f}%).")

print(f"\nVideos per year:")
print(df_merged['year'].value_counts().sort_index())

print("\n \nSample URLs from 100 random videos:")
sample_urls = df_merged.sample(100)['urls'].tolist()
all_urls = [url for urls in sample_urls if isinstance(urls, list) for url in urls]
print(f"Total URLs: {len(all_urls)}")
print("\nURL domains:")
from collections import Counter
domains = [url.split('/')[2] if len(url.split('/')) > 2 else url for url in all_urls]
print(Counter(domains).most_common(20))

The merge gave us 2,750,835 records, 1,249,165 records non matched (31.2%).

Videos per year:
year
2005         1
2006       598
2007      3600
2008      9165
2009     16335
2010     27756
2011     66000
2012     93545
2013    150076
2014    192231
2015    253825
2016    354232
2017    497749
2018    590846
2019    494876
Name: count, dtype: int64

 
Sample URLs from 100 random videos:


Total URLs: 582

URL domains:
[('bit.ly', 82), ('www.youtube.com', 81), ('twitter.com', 57), ('www.facebook.com', 55), ('amzn.to', 20), ('www.instagram.com', 18), ('goo.gl', 15), ('www.twitter.com', 12), ('instagram.com', 10), ('www.twitch.tv', 9), ('youtu.be', 9), ('plus.google.com', 6), ('facebook.com', 5), ('www.pinterest.com', 5), ('zacheven-esh.com', 5), ('www.patreon.com', 4), ('discord.gg', 3), ('soundcloud.com', 3), ('youtube.com', 3), ('www.mylifetime.com', 3)]


In [23]:
#What types of links? 
def has_patreon(urls):
    if not isinstance(urls, list):
        return False
    return any('patreon' in url.lower() for url in urls)

def has_paypal(urls):
    if not isinstance(urls, list):
        return False
    keywords = ['paypal', 'paypal.me']
    return any(any(kw in url.lower() for kw in keywords) for url in urls)

def has_kofi(urls):
    if not isinstance(urls, list):
        return False
    return any('ko-fi' in url.lower() or 'kofi' in url.lower() for url in urls)

def has_buymeacoffee(urls):
    if not isinstance(urls, list):
        return False
    return any('buymeacoffee' in url.lower() or 'buymeacoff.ee' in url.lower() for url in urls)

def has_gofundme(urls):
    if not isinstance(urls, list):
        return False
    return any('gofundme' in url.lower() for url in urls)

def has_kickstarter(urls):
    if not isinstance(urls, list):
        return False
    keywords = ['kickstarter', 'indiegogo']
    return any(any(kw in url.lower() for kw in keywords) for url in urls)

def has_merch(urls):
    """Merch platforms"""
    if not isinstance(urls, list):
        return False
    keywords = ['teespring', 'spreadshirt', 'redbubble', 'designbyhumans']
    return any(any(kw in url.lower() for kw in keywords) for url in urls)

def has_course(urls):
    """Course/digital product platforms"""
    if not isinstance(urls, list):
        return False
    keywords = ['gumroad', 'teachable', 'udemy', 'skillshare']
    return any(any(kw in url.lower() for kw in keywords) for url in urls)

def has_any_monetization(urls):
    """Any type of monetization link"""
    return (has_patreon(urls) or has_paypal(urls) or has_kofi(urls) or 
            has_buymeacoffee(urls) or has_gofundme(urls) or has_kickstarter(urls) or
            has_merch(urls) or has_course(urls))

df_merged['has_patreon'] = df_merged['urls'].apply(has_patreon)
df_merged['has_paypal'] = df_merged['urls'].apply(has_paypal)
df_merged['has_kofi'] = df_merged['urls'].apply(has_kofi)
df_merged['has_buymeacoffee'] = df_merged['urls'].apply(has_buymeacoffee)
df_merged['has_gofundme'] = df_merged['urls'].apply(has_gofundme)
df_merged['has_kickstarter'] = df_merged['urls'].apply(has_kickstarter)
df_merged['has_merch'] = df_merged['urls'].apply(has_merch)
df_merged['has_course'] = df_merged['urls'].apply(has_course)
df_merged['has_any_sponsor'] = df_merged['urls'].apply(has_any_monetization)

print(f"Patreon: {df_merged['has_patreon'].sum():,}")
print(f"PayPal: {df_merged['has_paypal'].sum():,}")
print(f"Ko-fi: {df_merged['has_kofi'].sum():,}")
print(f"Buy Me a Coffee: {df_merged['has_buymeacoffee'].sum():,}")
print(f"GoFundMe: {df_merged['has_gofundme'].sum():,}")
print(f"Kickstarter/Indiegogo: {df_merged['has_kickstarter'].sum():,}")
print(f"Merch: {df_merged['has_merch'].sum():,}")
print(f"Courses: {df_merged['has_course'].sum():,}")
print(f"\nANY monetization: {df_merged['has_any_sponsor'].sum():,}")
print(f"Percentage: {df_merged['has_any_sponsor'].sum()/len(df_merged)*100:.2f}%")

Patreon: 192,805
PayPal: 78,258
Ko-fi: 5,387
Buy Me a Coffee: 433
GoFundMe: 13,048
Kickstarter/Indiegogo: 3,252
Merch: 89,526
Courses: 6,334

ANY monetization: 308,601
Percentage: 11.22%


In [24]:
#finding the different categories
df_merged['has_donation'] = df_merged['has_paypal'] | df_merged['has_kofi'] | df_merged['has_buymeacoffee']
df_merged['has_crowdfunding'] = df_merged['has_kickstarter'] | df_merged['has_gofundme']
df_merged['has_products'] = df_merged['has_merch'] | df_merged['has_course']


In [25]:
# Aggregate by category and year
agg = df_merged.groupby(['categories', 'year']).agg(
    total=('display_id', 'count'),
    patreon_count=('has_patreon', 'sum'),
    donation_count=('has_donation', 'sum'),
    crowdfunding_count=('has_crowdfunding', 'sum'), 
    products_count=('has_products', 'sum'), 

).reset_index()

agg['membership_pct'] = (agg['patreon_count'] / agg['total']) * 100
agg['donation_pct'] = (agg['donation_count'] / agg['total']) * 100
agg['crowdfunding_pct'] = (agg['crowdfunding_count'] / agg['total']) * 100
agg['products_pct'] = (agg['products_count'] / agg['total']) * 100


In [26]:
#for the website
output = agg.to_dict(orient='records')
with open(repo_root / "analysis" / "vero_ana" / "sponsorships" / "sponsorship_data.json", "w") as f:
    json.dump(output, f, indent=2)

In [27]:
# There's a problem in 2018 (empty category). Investigating. 

year_2018 = agg[agg['year'] == 2018].copy()
#print(year_2018.sort_values('membership_pct', ascending=False)[['categories', 'total', 'patreon_count', 'membership_pct', 'donation_pct', 'products_pct']].head(15))

#patched all the way above by deleting all entries without a category


In [28]:
# Summary
print(f"Total aggregated records: {len(agg)}")
print(f"Categories: {sorted(agg['categories'].unique())}")

total_videos = df_merged.shape[0]
total_membership = df_merged['has_patreon'].sum()
total_donation = df_merged['has_donation'].sum()
total_products = df_merged['has_products'].sum()
total_any = df_merged['has_any_sponsor'].sum()

print(f"Total videos: {total_videos:,}")
print(f"With Patreon: {total_membership:,} ({total_membership/total_videos*100:.2f}%)")
print(f"With Donation: {total_donation:,} ({total_donation/total_videos*100:.2f}%)")
print(f"With Products: {total_products:,} ({total_products/total_videos*100:.2f}%)")
print(f"With any sponsorship: {total_any:,} ({total_any/total_videos*100:.2f}%)")

Total aggregated records: 209
Categories: ['Autos & Vehicles', 'Comedy', 'Education', 'Entertainment', 'Film & Animation', 'Gaming', 'Howto & Style', 'Music', 'News & Politics', 'Nonprofits & Activism', 'People & Blogs', 'Pets & Animals', 'Science & Technology', 'Sports', 'Travel & Events']
Total videos: 2,750,835
With Patreon: 192,805 (7.01%)
With Donation: 83,466 (3.03%)
With Products: 95,565 (3.47%)
With any sponsorship: 308,601 (11.22%)


In [29]:
#cumulative records 
cumulative_records = []

for year in sorted(agg['year'].unique()):   
    videos_up_to_year = df_merged[df_merged['year'] <= year]
    
    cum_agg = videos_up_to_year.groupby('categories').agg(
        total=('display_id', 'count'),
        patreon_count=('has_patreon', 'sum'),
        donation_count=('has_donation', 'sum'),
        crowdfunding_count=('has_crowdfunding', 'sum'),
        products_count=('has_products', 'sum')
    ).reset_index()
    
    cum_agg['membership_pct'] = (cum_agg['patreon_count'] / cum_agg['total']) * 100
    cum_agg['donation_pct'] = (cum_agg['donation_count'] / cum_agg['total']) * 100
    cum_agg['crowdfunding_pct'] = (cum_agg['crowdfunding_count'] / cum_agg['total']) * 100
    cum_agg['products_pct'] = (cum_agg['products_count'] / cum_agg['total']) * 100
    cum_agg['year'] = year
    
    cumulative_records.append(cum_agg)

agg_cumulative = pd.concat(cumulative_records, ignore_index=True)

output_cumulative = agg_cumulative.to_dict(orient='records')
with open(repo_root / "analysis" / "vero_ana" / "sponsorships" / "sponsorship_cumulative.json", "w") as f:
    json.dump(output_cumulative, f, indent=2)

In [30]:
# Find videos from 2007 with Patreon links [patreon only came out in 2013]

videos_2007_with_patreon = df_merged[
    (df_merged['year'] == 2007) & 
    (df_merged['has_patreon'] == True)
].copy()

print(f"Found {len(videos_2007_with_patreon)} videos from 2007 with Patreon links (patreon launched in 2013)")

if len(videos_2007_with_patreon) > 0:
    print("Sample videos:")
    for idx, row in videos_2007_with_patreon.head(10).iterrows():
        print(f"\nTitle: {row['title']}")
        print(f"Upload date: {row['upload_date']}")
        print(f"Category: {row['categories']}")
        print(f"Video ID: {row['display_id']}")
        print(f"Patreon URLs found:")
        patreon_urls = [url for url in row['urls'] if 'patreon' in url.lower()]
        for url in patreon_urls:
            print(f"  - {url}")
        print(f"YouTube link: https://youtube.com/watch?v={row['display_id']}")
else:
    print("No videos found (which would be correct!)")

# Also check the breakdown by year
print("\n" + "="*50)
print("Patreon links by upload year:")
patreon_by_year = df_merged[df_merged['has_patreon'] == True].groupby('year').size()
print(patreon_by_year.sort_index())

Found 16 videos from 2007 with Patreon links (patreon launched in 2013)
Sample videos:

Title: Art Zone - Cabin House
Upload date: 2007-07-15 00:00:00
Category: Film & Animation
Video ID: CSBARYAZso8
Patreon URLs found:
  - https://www.patreon.com/ArtZone
YouTube link: https://youtube.com/watch?v=CSBARYAZso8

Title: The Spiral Dance
Upload date: 2007-03-14 00:00:00
Category: Nonprofits & Activism
Video ID: y_zhZeRNfSg
Patreon URLs found:
  - https://www.patreon.com/BernardAlvarez
YouTube link: https://youtube.com/watch?v=y_zhZeRNfSg

Title: Clean Arpeggios
Upload date: 2007-12-08 00:00:00
Category: Music
Video ID: S8hESGwY0V8
Patreon URLs found:
  - https://www.patreon.com/METALHAWK
YouTube link: https://youtube.com/watch?v=S8hESGwY0V8

Title: my new heavy metal riff
Upload date: 2007-11-26 00:00:00
Category: Music
Video ID: giIPnbUkp0U
Patreon URLs found:
  - https://www.patreon.com/METALHAWK
YouTube link: https://youtube.com/watch?v=giIPnbUkp0U

Title: Getting to know people in the s